Amazon Locker is a self-service package pickup system. A delivery driver deposits a package into an available compartment, the system generates an access token, and the customer uses that code to retrieve their package.


### QUESTIONS:

Core functions:

- How many compartments are there? 
- Sizes of compartments?
- Does driver need authentication/authorization to open the compartments? How is that done?
- What is the work flow for the driver to deliver the page? Does she just scan a code on the package before depositing it into a compartment?
- Is there expiration if package is not picked up?
- How is the token delivered to the customer?

Error conditions:
- What happens when all compartments are used? Or no right size of compartments?
- Driver opens a compartment but doesn't close it?
- What happens when customer enters wrong token, expired token or claimed token?
- What if package is not picked up?
- What about malicious attack, like force opening a compartment or many wrong token attempts?

Future extension or scope:
- UI?
- tracking? auditing?
- adding more compartments?
- customers use the locker to return packages?

### REQUIREMENTS

- Locker containing compartments of different sizes
- Driver scans code on package (my assumption), system will open an available compartment of matching size. If there is none available, error will be returned. Driver will take package back.
- Once a package is deposit, an access token will be created. Access token is unique per package.
- Customer uses token to open compartment and retrieve package. If customer enters a wrong token, expired token or used token, error is returned.
- Token will be expired after 7 days if package is not picked up. Driver will pick up expired package.

Out of Scope:
- UI
- Logic to encode/decode the token
- Tracking, auditing
- Token delivery to customers
- Mechanism to handle malicious attacks (force open or token guessing)
- Using the locker to return packages

### IMPROVEMENTS

- use number to identify each requirements
- has a main subject for each requirement and action it performs or operation on it.

### ENTITIES AND RELATIONSHIPS

Locker - the entry point and the orchestrator. It holds a collection of compartments and token mapping. Owns the package deposit and pickup.
    
Compartment - has size and state (occupied or available)

Package - has size

Token - has an access code, creation date and reference to compartment it maps to. Owns expiry validation.

### IMPROVEMENTS
- Should describe my thinking process, how I come up with the entities
- There is no need for a pakage class. All we need is an enum for size.

### CLASS DESIGN

Class Locker:
    - compartments: Compartment[]  //all compartments regardless of state
    - tokenMapping: {token.AccessCode, compartment}  //mapping for occupied compartments
    DepositPackage(Package) -> Token
    PickupPackage(Token)

Class Compartment:
    - CompartmentState: Enum(available, occupied)
    - Size: Enum(Small, Medium, Large)
    GetSize() -> CompartmentSize
    IsAvailable() -> Boolean
    Open(Token) -> Boolean
    
Class Token:
    - AccessCode
    - CreationDate
    - Compartment
    CreateNewToken(Compartment) -> Token
    IsValid() -> Boolean

### IMPROVEMENTS

- Need to include constructor for each class and the parameters for the constructor.
- Include data type for each properties

Locker Class
- Constructor missing
- tokenMapping should map the access code (string) to the token object
- The pickup method should pass in the access code (string) instead of the token object. Consider what the input is from the customer.
- I miss function for opening lockers with expired packages 

Compartment Class
- Constructor missing
- a boolean for occupacy is enough
- Open function doesn't need token to pass in
- need markOccupied() and markFree()

AccessToken Class
- missing constructor
- should have the expiration timestamp instead of creation which is determined by the Locker (orchestrator)
- Create new access code is not something the Token Class' job, it is provided from the caller. The Token class only needs to contain the access code.
- missing getCompartment(), since the Token class is responsible to keep track of all the state and information related to the access token. The Locker class doesn't own this relationship.
- missing getCode() method. Why is it needed??